In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import nibabel as nib
import numpy as np
from templateflow import api as tflow
import matplotlib.pyplot as plt
from nilearn.plotting import plot_glass_brain

from lib import decoders, decoders_by_coverage


In [ ]:
base_dir = Path(os.getcwd()).parent
space = "MNI152NLin2009cAsym"
decoder_path = base_dir / f"space-{space}"
figure_path = base_dir / "figures"


In [ ]:
mni152_template_path = tflow.get("MNI152NLin2009cAsym", resolution=1, desc="brain", suffix="T1w")


In [ ]:
names = decoders_by_coverage()

Define a function to plot the glass brain representation of the decoder.

In [ ]:
def plot_decoder_glass_brain(decoder_path, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    else:
        fig = ax.get_figure()
    plot_glass_brain(
        decoder_path,
        # bg_img=mni152_template_path,
        # title=decoder_name,
        plot_abs=False,
        vmin=-0.010,
        vmax=0.010,
        colorbar=True,
        cmap='RdBu_r',
        black_bg=False,
        axes=ax,
    )
    return fig, ax


In [ ]:
def orientation_string_from_affine(affine):

    x = "R" if affine[0, 0] >= 0.0 else "L"
    y = "A" if affine[1, 1] >= 0.0 else "P"
    z = "S" if affine[2, 2] >= 0.0 else "I"
    return f"{x}{y}{z}"


In [ ]:
for coverage in sorted(set([d['coverage'] for d in decoders.values()])):
    # Create half as many rows as decoders, but need at least one.
    nrows = max(1, int((len(names[coverage]) + 0.5) / 2))
    # The figure should be a bit under 2" per row, plus a half inch for the title.
    vsize = nrows * 1.7 + 1.5
    print(f"Plotting {len(names[coverage])} decoders "
          f"({nrows} rows, {vsize} inches) for {coverage} coverage")

    fig_gb, axes_gb = plt.subplots(ncols=2, nrows=nrows, figsize=(12, vsize))
    fig_gb.suptitle(coverage)
    row, col = 0, 0
    for name in names[coverage]:
        d = decoders[name]
        filename = (
            f"{name.lower().replace(' ', '').replace('(alt)', '')}_"
            f"{d['coverage'].lower().replace(' ', '')}_"
            f"space-{space}_weights.nii.gz")
        _ax = axes_gb[row, col] if nrows > 1 else axes_gb[col]
        decoder_img = nib.load(decoder_path / filename)
        d['min_weight'] = np.min(decoder_img.get_fdata())
        d['mean_weight'] = np.mean(decoder_img.get_fdata())
        d['max_weight'] = np.max(decoder_img.get_fdata())
        d['num_non_zero_voxels'] = np.sum(decoder_img.get_fdata() != 0.0)
        d['num_voxels'] = len(decoder_img.get_fdata().ravel())
        d['x_dim'], d['y_dim'], d['z_dim'] = decoder_img.header.get_zooms()
        range_str = f"({d['min_weight']:.4f}, {d['mean_weight']:.4f}, {d['max_weight']:.4f})"
        print(f"{filename:<50} {str(decoder_img.shape):<14} {range_str} "
              f"({d['num_non_zero_voxels']:,}/{d['num_voxels']:,} non-zero)")
        d['glass_brain_fig'], _ = plot_decoder_glass_brain(
            decoder_path / filename, ax=_ax
        )
        _ax.set_title(f"{name} ({d['num_non_zero_voxels']:,} "
                     f"[{d['x_dim']:.1f}x{d['y_dim']:.1f}x{d['z_dim']:.1f}] voxels) "
                     f"({orientation_string_from_affine(decoder_img.affine)})")
        _ax.set_facecolor("skyblue")
        col += 1
        if col == 2:
            col = 0
            row += 1
    while row < nrows:
        _ax = axes_gb[row, col] if nrows > 1 else axes_gb[col]
        _ax.axis('off')
        col += 1
        if col == 2:
            col = 0
            row += 1
    fig_gb.savefig(base_dir / "figures" / f"glass_brain_{coverage.replace(' ', '_')}_decoders.png")
